retrival notebook

Load historical Amazon conversations + golden set
Create embeddings with all-MiniLM-L6-v2
Build the FAISS index
Retrieve Top-K historical cases
Add relevance labels for a small subset of golden examples
Calculate Recall@K and MRR
Inspect retrieval failures

In [3]:
# ============================================================
# RETRIEVAL EVALUATION — CELL 1
# Load historical cases + golden set
# ============================================================

import pandas as pd
import numpy as np

# Go one level up from notebooks/ to the project root
history = pd.read_csv(
    "../data/processed/amazon_conversations.csv"
)

golden = pd.read_csv(
    "../data/golden/golden_set.csv"
)

print("Historical conversations:", history.shape)
print("Golden set:", golden.shape)

print("\nHistorical columns:")
print(history.columns.tolist())

print("\nGolden columns:")
print(golden.columns.tolist())

print("\nGolden intents:")
print(golden["intent"].value_counts())

Historical conversations: (76799, 3)
Golden set: (200, 5)

Historical columns:
['root_tweet_id', 'thread_size', 'conversation']

Golden columns:
['root_tweet_id', 'thread_size', 'conversation', 'intent', 'annotation_notes']

Golden intents:
intent
Delivery Issue                    68
Other / Unclear                   23
Damaged / Wrong / Missing Item    21
Digital Content                   15
Prime Membership                  13
Return / Refund                   13
Promotion / Gift Card / Credit    12
Payment / Billing                 11
Account / Access / Security        8
Order Management                   8
Product / Device Support           8
Name: count, dtype: int64


In [4]:
# ============================================================
# RETRIEVAL EVALUATION — CELL 2
# Create embeddings for historical conversations
# ============================================================

from sentence_transformers import SentenceTransformer

# Load the same embedding model used in classifier evaluation
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# Create one semantic embedding per historical conversation
history_embeddings = embedding_model.encode(
    history["conversation"].tolist(),
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
)

print("Embedding shape:", history_embeddings.shape)
print("Number of historical cases:", len(history))
print("Embedding dimension:", history_embeddings.shape[1])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/1200 [00:00<?, ?it/s]

Embedding shape: (76799, 384)
Number of historical cases: 76799
Embedding dimension: 384


In [6]:
# ============================================================
# 
# Build FAISS index
# ============================================================

import faiss

# Embeddings are normalized, so inner product = cosine similarity
dimension = history_embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)

# Add historical conversation embeddings
index.add(
    np.asarray(history_embeddings, dtype="float32")
)

print("FAISS index built successfully!")
print("Index size:", index.ntotal)
print("Vector dimension:", index.d)

FAISS index built successfully!
Index size: 76799
Vector dimension: 384


In [7]:
# ============================================================
# RETRIEVAL EVALUATION 
# Test Top-K retrieval on one golden example
# ============================================================

# Take the first golden example
query_text = golden.loc[0, "conversation"]

# Create embedding for the query
query_embedding = embedding_model.encode(
    [query_text],
    normalize_embeddings=True
).astype("float32")

# Retrieve top 5 historical conversations
scores, indices = index.search(query_embedding, 5)

print("QUERY:")
print(query_text)

print("\n" + "=" * 80)
print("TOP 5 HISTORICAL CASES")
print("=" * 80)

for rank, (score, idx) in enumerate(zip(scores[0], indices[0]), start=1):
    print(f"\nRank {rank} | Similarity: {score:.4f}")
    print("-" * 60)
    print(history.iloc[idx]["conversation"])

QUERY:
CUSTOMER: Well… that’s one way to deliver a Pyrex measuring cup… #shattered @115821 @118706 https://t.co/82CziNeRrH

AMAZON: @410862 Oh no, Mason! I'm terribly sorry for the poor delivery experience! Who was the carrier assigned to this shipment? ^SA

TOP 5 HISTORICAL CASES

Rank 1 | Similarity: 1.0000
------------------------------------------------------------
CUSTOMER: Well… that’s one way to deliver a Pyrex measuring cup… #shattered @115821 @118706 https://t.co/82CziNeRrH

AMAZON: @410862 Oh no, Mason! I'm terribly sorry for the poor delivery experience! Who was the carrier assigned to this shipment? ^SA

Rank 2 | Similarity: 0.6102
------------------------------------------------------------
CUSTOMER: Climbed out of my window today because @115821 trapped a package between the opening of my door and a pot. Who do I yell at !!!! 👺

AMAZON: @291705 Thanks for contacting us on here. Can you confirm the carrier that delivered the order here: https://t.co/q4LAMZ3tbE?  ^GG

Rank 

In [8]:
# ============================================================
# RETRIEVAL EVALUATION 
# Remove Golden Set from retrieval corpus
# ============================================================

# Golden conversation IDs
golden_ids = set(
    golden["root_tweet_id"].astype(int)
)

# Keep only historical conversations not present in Golden Set
history_mask = ~history["root_tweet_id"].astype(int).isin(golden_ids)

history_eval = history[history_mask].reset_index(drop=True)
history_embeddings_eval = history_embeddings[history_mask.values]

print("Original historical cases:", len(history))
print("Golden cases removed:", len(history) - len(history_eval))
print("Leakage-safe retrieval cases:", len(history_eval))
print("Embedding shape:", history_embeddings_eval.shape)

Original historical cases: 76799
Golden cases removed: 200
Leakage-safe retrieval cases: 76599
Embedding shape: (76599, 384)


In [9]:
# ============================================================
# RETRIEVAL EVALUATION 
# Build leakage-safe FAISS index
# ============================================================

import faiss

# Embeddings are normalized, so inner product = cosine similarity
dimension = history_embeddings_eval.shape[1]

eval_index = faiss.IndexFlatIP(dimension)

# Add only non-golden historical conversations
eval_index.add(
    np.asarray(history_embeddings_eval, dtype="float32")
)

print("Leakage-safe FAISS index built!")
print("Index size:", eval_index.ntotal)
print("Vector dimension:", eval_index.d)

Leakage-safe FAISS index built!
Index size: 76599
Vector dimension: 384


In [10]:
# ============================================================
# RETRIEVAL EVALUATION 
# Test leakage-safe retrieval
# ============================================================

# Use the first golden example as the query
query_text = golden.loc[0, "conversation"]

# Create query embedding
query_embedding = embedding_model.encode(
    [query_text],
    normalize_embeddings=True
).astype("float32")

# Retrieve top 5 from leakage-safe index
scores, indices = eval_index.search(query_embedding, 5)

print("QUERY:")
print(query_text)

print("\n" + "=" * 80)
print("TOP 5 HISTORICAL CASES (LEAKAGE-SAFE)")
print("=" * 80)

for rank, (score, idx) in enumerate(zip(scores[0], indices[0]), start=1):
    print(f"\nRank {rank} | Similarity: {score:.4f}")
    print("-" * 60)
    print(history_eval.iloc[idx]["conversation"][:1000])

QUERY:
CUSTOMER: Well… that’s one way to deliver a Pyrex measuring cup… #shattered @115821 @118706 https://t.co/82CziNeRrH

AMAZON: @410862 Oh no, Mason! I'm terribly sorry for the poor delivery experience! Who was the carrier assigned to this shipment? ^SA

TOP 5 HISTORICAL CASES (LEAKAGE-SAFE)

Rank 1 | Similarity: 0.6102
------------------------------------------------------------
CUSTOMER: Climbed out of my window today because @115821 trapped a package between the opening of my door and a pot. Who do I yell at !!!! 👺

AMAZON: @291705 Thanks for contacting us on here. Can you confirm the carrier that delivered the order here: https://t.co/q4LAMZ3tbE?  ^GG

Rank 2 | Similarity: 0.6058
------------------------------------------------------------
CUSTOMER: @AmazonHelp Thank you so much! The new one looks so much better! Great job!

AMAZON: @429607 Glad to hear it arrived undamaged and in good condition! Thanks for keeping us in the loop on this! ^BV

CUSTOMER: Ordered a Bob Ross pop f

In [11]:
# ============================================================
# RETRIEVAL EVALUATION — CELL 8
# Retrieve Top-5 cases for all Golden examples
# ============================================================

# Create embeddings for all golden conversations
golden_embeddings = embedding_model.encode(
    golden["conversation"].tolist(),
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True
).astype("float32")

# Retrieve top 5 historical cases for every golden example
scores, indices = eval_index.search(
    golden_embeddings,
    5
)

print("Golden queries:", len(golden))
print("Retrieved per query:", indices.shape[1])
print("Scores shape:", scores.shape)
print("Indices shape:", indices.shape)

Batches:   0%|          | 0/7 [00:00<?, ?it/s]

Golden queries: 200
Retrieved per query: 5
Scores shape: (200, 5)
Indices shape: (200, 5)


In [12]:
# ============================================================
# RETRIEVAL — CELL 10
# Load 10K historical retrieval corpus
# ============================================================

retrieval_data = pd.read_csv(
    "../data/processed/dev_pseudo_labeled.csv"
)

print("Retrieval corpus:", retrieval_data.shape)
print("Columns:", retrieval_data.columns.tolist())

print("\nIntent distribution:")
print(retrieval_data["pseudo_intent"].value_counts())

Retrieval corpus: (10000, 5)
Columns: ['root_tweet_id', 'thread_size', 'conversation', 'pseudo_intent', 'reviewed']

Intent distribution:
pseudo_intent
Delivery Issue                    3900
Other / Unclear                   2173
Account / Access / Security        627
Product / Device Support           586
Order Management                   583
Payment / Billing                  519
Return / Refund                    507
Damaged / Wrong / Missing Item     494
Digital Content                    354
Prime Membership                   157
Promotion / Gift Card / Credit     100
Name: count, dtype: int64


In [14]:
# ============================================================
# RETRIEVAL — CELL 11
# Create embeddings for 10K retrieval corpus
# ============================================================

retrieval_embeddings = embedding_model.encode(
    retrieval_data["conversation"].tolist(),
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
).astype("float32")

print("Embedding shape:", retrieval_embeddings.shape)
print("Retrieval cases:", len(retrieval_data))
print("Embedding dimension:", retrieval_embeddings.shape[1])

Batches:   0%|          | 0/157 [00:00<?, ?it/s]

Embedding shape: (10000, 384)
Retrieval cases: 10000
Embedding dimension: 384


In [15]:
# ============================================================
# RETRIEVAL — CELL 12
# Build final 10K FAISS index
# ============================================================

import faiss

dimension = retrieval_embeddings.shape[1]

retrieval_index = faiss.IndexFlatIP(dimension)

retrieval_index.add(retrieval_embeddings)

print("FAISS retrieval index built successfully!")
print("Index size:", retrieval_index.ntotal)
print("Vector dimension:", retrieval_index.d)

FAISS retrieval index built successfully!
Index size: 10000
Vector dimension: 384


In [16]:
# ============================================================
# RETRIEVAL — CELL 13
# Test retrieval on one Golden example
# ============================================================

# Select one Golden example
query = golden.iloc[0]["conversation"]

# Create query embedding
query_embedding = embedding_model.encode(
    [query],
    normalize_embeddings=True
).astype("float32")

# Retrieve top 5 similar historical cases
scores, indices = retrieval_index.search(
    query_embedding,
    5
)

print("QUERY")
print("=" * 80)
print(query)

print("\nTOP 5 HISTORICAL CASES")
print("=" * 80)

for rank, (score, idx) in enumerate(
    zip(scores[0], indices[0]), start=1
):
    case = retrieval_data.iloc[idx]

    print(f"\nRank {rank}")
    print(f"Similarity: {score:.4f}")
    print(f"Historical Intent: {case['pseudo_intent']}")
    print("-" * 60)
    print(case["conversation"][:1200])

QUERY
CUSTOMER: Well… that’s one way to deliver a Pyrex measuring cup… #shattered @115821 @118706 https://t.co/82CziNeRrH

AMAZON: @410862 Oh no, Mason! I'm terribly sorry for the poor delivery experience! Who was the carrier assigned to this shipment? ^SA

TOP 5 HISTORICAL CASES

Rank 1
Similarity: 0.5834
Historical Intent: Delivery Issue
------------------------------------------------------------
CUSTOMER: @115821 delivery is great &amp; all until the driver decides it’s raining too hard, falsely marks your packages delivered, &amp; leaves😠 @AmazonHelp

AMAZON: @203162 Sorry for the recent problems with your delivery! Just to clarify, who was the carrier? Check here: https://t.co/Y5jpI9gRhE

Rank 2
Similarity: 0.5769
Historical Intent: Damaged / Wrong / Missing Item
------------------------------------------------------------
CUSTOMER: @AmazonHelp I haven’t recieved an order that shows as “delivered” I can’t see where I can contact? It just says delivered. I recieved a different mug

In [17]:
# ============================================================
# GENERATION — CELL 14
# Prepare retrieved cases for the LLM
# ============================================================

def retrieve_cases(query_text, top_k=5):
    """
    Retrieve the most similar historical Amazon cases.
    """

    # Convert customer message into an embedding
    query_embedding = embedding_model.encode(
        [query_text],
        normalize_embeddings=True
    ).astype("float32")

    # Search FAISS
    scores, indices = retrieval_index.search(
        query_embedding,
        top_k
    )

    results = []

    for rank, (score, idx) in enumerate(
        zip(scores[0], indices[0]),
        start=1
    ):
        case = retrieval_data.iloc[idx]

        results.append({
            "rank": rank,
            "similarity": float(score),
            "intent": case["pseudo_intent"],
            "conversation": case["conversation"]
        })

    return results


# Test the retrieval function
results = retrieve_cases(
    golden.iloc[0]["conversation"],
    top_k=5
)

print("Retrieved cases:", len(results))

for case in results:
    print(
        f"\nRank {case['rank']} | "
        f"Similarity: {case['similarity']:.4f} | "
        f"Intent: {case['intent']}"
    )

Retrieved cases: 5

Rank 1 | Similarity: 0.5834 | Intent: Delivery Issue

Rank 2 | Similarity: 0.5769 | Intent: Damaged / Wrong / Missing Item

Rank 3 | Similarity: 0.5742 | Intent: Damaged / Wrong / Missing Item

Rank 4 | Similarity: 0.5737 | Intent: Delivery Issue

Rank 5 | Similarity: 0.5639 | Intent: Delivery Issue


In [25]:
import os
from openai import OpenAI
from dotenv import load_dotenv

# Load .env
load_dotenv(r"K:\projects\sales agent hiver\.env", override=True)

client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

MODEL_NAME = os.getenv("GROQ_MODEL", "openai/gpt-oss-20b")

print("Groq client ready!")
print("Model:", MODEL_NAME)

Groq client ready!
Model: openai/gpt-oss-20b


In [26]:
def generate_reply(customer_message, intent, retrieved_cases):
    # Build historical evidence
    evidence = ""

    for case in retrieved_cases:
        evidence += f"""
CASE {case['rank']}
Similarity: {case['similarity']:.3f}
Historical Intent: {case['intent']}

{case['conversation']}

---
"""

    prompt = f"""
You are an Amazon customer support assistant.

Your task is to draft a helpful customer support reply.

CUSTOMER MESSAGE:
{customer_message}

CLASSIFIED INTENT:
{intent}

HISTORICAL SUPPORT CASES:
{evidence}

RULES:
1. Use the historical cases as evidence for how Amazon handled similar issues.
2. Do not invent policies, refunds, dates, links, guarantees, or actions.
3. If the historical evidence is insufficient, say that the customer should contact support for further assistance.
4. Do not mention that you are using historical cases.
5. Be concise, polite, and professional.
6. Directly address the customer's problem.
7. Do not claim that an action has already been taken unless the evidence supports it.
8. Return only the customer-facing reply.

Draft the reply:
"""

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {
                "role": "system",
                "content": "You are a careful, evidence-grounded customer support assistant."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.2,
        max_tokens=300
    )

    return response.choices[0].message.content.strip()


print("Reply generator ready!")

Reply generator ready!


In [27]:
# Use the first Golden Set example
test_row = golden.iloc[0]

customer_message = test_row["conversation"]

# Retrieve similar historical Amazon cases
retrieved_cases = retrieve_cases(
    customer_message,
    top_k=5
)

# Generate grounded reply
reply = generate_reply(
    customer_message=customer_message,
    intent=test_row["intent"],
    retrieved_cases=retrieved_cases
)

print("CUSTOMER:")
print(customer_message)

print("\nCLASSIFIED INTENT:")
print(test_row["intent"])

print("\nGENERATED REPLY:")
print(reply)

print("\nTOP RETRIEVED CASES:")
for case in retrieved_cases:
    print(
        f"Rank {case['rank']} | "
        f"Similarity: {case['similarity']:.3f} | "
        f"Intent: {case['intent']}"
    )

CUSTOMER:
CUSTOMER: Well… that’s one way to deliver a Pyrex measuring cup… #shattered @115821 @118706 https://t.co/82CziNeRrH

AMAZON: @410862 Oh no, Mason! I'm terribly sorry for the poor delivery experience! Who was the carrier assigned to this shipment? ^SA

CLASSIFIED INTENT:
Delivery Issue

GENERATED REPLY:
I’m sorry your Pyrex measuring cup arrived shattered.  
Was this item shipped by Amazon or a third‑party seller? If it was shipped by Amazon, you can reach out to us via phone or chat to discuss options: https://t.co/Tz2mbn87WY.

TOP RETRIEVED CASES:
Rank 1 | Similarity: 0.583 | Intent: Delivery Issue
Rank 2 | Similarity: 0.577 | Intent: Damaged / Wrong / Missing Item
Rank 3 | Similarity: 0.574 | Intent: Damaged / Wrong / Missing Item
Rank 4 | Similarity: 0.574 | Intent: Delivery Issue
Rank 5 | Similarity: 0.564 | Intent: Delivery Issue


In [28]:
def generate_reply(customer_message, intent, retrieved_cases):
    evidence = ""

    for case in retrieved_cases:
        evidence += f"""
CASE {case['rank']}
Similarity: {case['similarity']:.3f}
Historical Intent: {case['intent']}

{case['conversation']}

---
"""

    prompt = f"""
You are an Amazon customer support assistant.

CUSTOMER MESSAGE:
{customer_message}

CLASSIFIED INTENT:
{intent}

HISTORICAL SUPPORT CASES:
{evidence}

Your task is to draft a concise customer-facing support reply grounded ONLY in the historical cases.

STRICT RULES:
1. Use the historical cases as evidence for how similar issues were handled.
2. Do NOT invent policies, refunds, replacements, links, phone numbers, guarantees,
   shipping methods, seller types, or other facts.
3. Do NOT create or reproduce URLs unless a URL is explicitly present in the
   historical evidence and is directly relevant.
4. Do NOT claim that Amazon has taken an action unless the evidence supports it.
5. If the evidence does not provide a specific resolution, acknowledge the issue
   and ask an appropriate clarifying question or direct the customer to support.
6. Be concise, polite, professional, and helpful.
7. Do not mention historical cases, retrieval, similarity, AI, or classification.
8. Return ONLY the customer-facing reply.

Draft the reply:
"""

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {
                "role": "system",
                "content": "You are a careful, evidence-grounded customer support assistant."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.2,
        max_tokens=300
    )

    return response.choices[0].message.content.strip()


print("Grounded reply generator updated!")

Grounded reply generator updated!


In [43]:
def escalation_policy(customer_message, intent, retrieved_cases):
    text = customer_message.lower()

    # High-risk account / financial / legal / safety signals
    high_risk_keywords = [
        "hacked",
        "hack",
        "account stolen",
        "unauthorized",
        "unauthorised",
        "without my permission",
        "fraud",
        "scam",
        "stolen",
        "chargeback",
        "lawsuit",
        "legal",
        "police",
        "danger",
        "injured",
        "unsafe"
    ]

    # Customer explicitly requests a human
    human_request_keywords = [
        "human",
        "agent",
        "representative",
        "real person",
        "speak to someone",
        "talk to someone",
        "speak with someone"
    ]

    max_similarity = max(
        [case["similarity"] for case in retrieved_cases],
        default=0.0
    )

    # High-risk issue
    if any(keyword in text for keyword in high_risk_keywords):
        return {
            "decision": "ESCALATE",
            "reason": "High-risk account, financial, legal, or safety-related issue."
        }

    # Explicit human request
    if any(keyword in text for keyword in human_request_keywords):
        return {
            "decision": "ESCALATE",
            "reason": "Customer explicitly requested human assistance."
        }

    # Insufficient historical evidence
    if max_similarity < 0.50:
        return {
            "decision": "ESCALATE",
            "reason": "Insufficient similarity to historical support cases."
        }

    return {
        "decision": "AUTO-HANDLE",
        "reason": "No escalation trigger detected and relevant historical evidence was retrieved."
    }


print("Escalation policy updated!")

Escalation policy updated!


In [31]:
def run_agent(customer_message, intent, top_k=5):
    # 1. Retrieve similar historical support cases
    retrieved_cases = retrieve_cases(
        customer_message,
        top_k=top_k
    )

    # 2. Generate evidence-grounded reply
    reply = generate_reply(
        customer_message=customer_message,
        intent=intent,
        retrieved_cases=retrieved_cases
    )

    # 3. Decide whether to escalate
    escalation = escalation_policy(
        customer_message=customer_message,
        intent=intent,
        retrieved_cases=retrieved_cases
    )

    return {
        "customer_message": customer_message,
        "intent": intent,
        "retrieved_cases": retrieved_cases,
        "reply": reply,
        "escalation": escalation
    }


print("SupportIQ agent ready!")

SupportIQ agent ready!


In [32]:
# Test with the first Golden Set example
test_row = golden.iloc[0]

result = run_agent(
    customer_message=test_row["conversation"],
    intent=test_row["intent"],
    top_k=5
)

print("=" * 70)
print("CUSTOMER")
print("=" * 70)
print(result["customer_message"])

print("\n" + "=" * 70)
print("INTENT")
print("=" * 70)
print(result["intent"])

print("\n" + "=" * 70)
print("RETRIEVED CASES")
print("=" * 70)

for case in result["retrieved_cases"]:
    print(
        f"Rank {case['rank']} | "
        f"Similarity: {case['similarity']:.3f} | "
        f"Intent: {case['intent']}"
    )

print("\n" + "=" * 70)
print("GENERATED REPLY")
print("=" * 70)
print(result["reply"])

print("\n" + "=" * 70)
print("ESCALATION")
print("=" * 70)
print("Decision:", result["escalation"]["decision"])
print("Reason:", result["escalation"]["reason"])

CUSTOMER
CUSTOMER: Well… that’s one way to deliver a Pyrex measuring cup… #shattered @115821 @118706 https://t.co/82CziNeRrH

AMAZON: @410862 Oh no, Mason! I'm terribly sorry for the poor delivery experience! Who was the carrier assigned to this shipment? ^SA

INTENT
Delivery Issue

RETRIEVED CASES
Rank 1 | Similarity: 0.583 | Intent: Delivery Issue
Rank 2 | Similarity: 0.577 | Intent: Damaged / Wrong / Missing Item
Rank 3 | Similarity: 0.574 | Intent: Damaged / Wrong / Missing Item
Rank 4 | Similarity: 0.574 | Intent: Delivery Issue
Rank 5 | Similarity: 0.564 | Intent: Delivery Issue

GENERATED REPLY
I’m terribly sorry for the poor delivery experience! Who was the carrier assigned to this shipment?

ESCALATION
Decision: AUTO-HANDLE
Reason: No high-risk signal or explicit human request, and relevant historical evidence was retrieved.


In [33]:
test_messages = [
    "My package was supposed to arrive yesterday but I still haven't received it.",
    "Someone hacked my Amazon account and I see unauthorized charges.",
    "I want to speak to a human agent about my order.",
    "Can you help me with this completely unrelated question about my vacation?"
]

for i, message in enumerate(test_messages, start=1):
    print("\n" + "=" * 70)
    print(f"TEST CASE {i}")
    print("=" * 70)
    
    result = run_agent(
        customer_message=message,
        intent="Other / Unclear",
        top_k=5
    )
    
    print("\nCUSTOMER:")
    print(message)
    
    print("\nREPLY:")
    print(result["reply"])
    
    print("\nESCALATION:")
    print(result["escalation"]["decision"])
    
    print("REASON:")
    print(result["escalation"]["reason"])
    
    print("\nTOP SIMILARITY:")
    print(f"{result['retrieved_cases'][0]['similarity']:.3f}")


TEST CASE 1

CUSTOMER:
My package was supposed to arrive yesterday but I still haven't received it.

REPLY:
I’m sorry your package hasn’t arrived yet. Please reach out to us here so we can look into this delivery: https://t.co/hApLpMlfHN

ESCALATION:
AUTO-HANDLE
REASON:
No high-risk signal or explicit human request, and relevant historical evidence was retrieved.

TOP SIMILARITY:
0.803

TEST CASE 2

CUSTOMER:
Someone hacked my Amazon account and I see unauthorized charges.

REPLY:


ESCALATION:
ESCALATE
REASON:
High-risk account, financial, legal, or safety-related issue.

TOP SIMILARITY:
0.835

TEST CASE 3

CUSTOMER:
I want to speak to a human agent about my order.

REPLY:


ESCALATION:
ESCALATE
REASON:
Customer explicitly requested human assistance.

TOP SIMILARITY:
0.619

TEST CASE 4

CUSTOMER:
Can you help me with this completely unrelated question about my vacation?

REPLY:
I’m sorry, but I’m not sure how to help with that. Could you please give me a bit more detail about your va

In [34]:
print("Available classifier-related objects:")

for name in [
    "hybrid_model",
    "hybrid_classifier",
    "tfidf_vectorizer",
    "embedding_model",
    "semantic_model",
    "models",
    "predict_intent",
    "classify_intent"
]:
    if name in globals():
        obj = globals()[name]
        print(f"✓ {name}: {type(obj).__name__}")
    else:
        print(f"✗ {name}: not found")

Available classifier-related objects:
✗ hybrid_model: not found
✗ hybrid_classifier: not found
✗ tfidf_vectorizer: not found
✓ embedding_model: SentenceTransformer
✗ semantic_model: not found
✗ models: not found
✗ predict_intent: not found
✗ classify_intent: not found


In [35]:
import json
from pathlib import Path

nb_path = Path("05_classifier_evaluation.ipynb")

with open(nb_path, "r", encoding="utf-8") as f:
    nb = json.load(f)

print("Notebook loaded:", nb_path)
print("Number of cells:", len(nb["cells"]))

for i, cell in enumerate(nb["cells"]):
    text = "".join(cell.get("source", []))
    
    if any(word in text.lower() for word in [
        "hybrid",
        "logisticregression",
        "tfidf",
        "semantic"
    ]):
        print(f"\n--- Cell {i} ---")
        print(text[:1500])

Notebook loaded: 05_classifier_evaluation.ipynb
Number of cells: 24

--- Cell 3 ---
# ============================================================
# BASELINE 2 — TF-IDF + LOGISTIC REGRESSION
# CELL 1: LOAD DATA
# ============================================================

import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

DEV_PATH = "../data/processed/dev_pseudo_labeled.csv"
GOLDEN_PATH = "../data/golden/golden_set.csv"

# ------------------------------------------------------------
# Load datasets
# ------------------------------------------------------------

dev = pd.read_csv(DEV_PATH)
golden = pd.read_csv(GOLDEN_PATH)

# Clean labels
dev["pseudo_intent"] = dev["pseudo_intent"].str.strip()
golden["intent"] = golden["intent"].str.strip()

# ---------------------------

In [37]:
def extract_customer_text(conversation):
    lines = str(conversation).splitlines()

    customer_messages = []

    for line in lines:
        if line.startswith("CUSTOMER:"):
            message = line.replace("CUSTOMER:", "", 1).strip()
            customer_messages.append(message)

    return " ".join(customer_messages)


print("Customer text extractor ready!")

Customer text extractor ready!


In [38]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from scipy.sparse import hstack, csr_matrix

# Load development data
dev = pd.read_csv("../data/processed/dev_pseudo_labeled.csv")
golden = pd.read_csv("../data/golden/golden_set.csv")

# Extract customer messages only
dev["customer_text"] = dev["conversation"].apply(extract_customer_text)
golden["customer_text"] = golden["conversation"].apply(extract_customer_text)

# Same 8K / 2K split used in classifier evaluation
from sklearn.model_selection import train_test_split

dev_train, dev_test = train_test_split(
    dev,
    test_size=2000,
    random_state=42,
    stratify=dev["pseudo_intent"]
)

# ------------------------------------------------------------
# TF-IDF
# ------------------------------------------------------------

tfidf_vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True,
    max_features=50000
)

X_train_tfidf = tfidf_vectorizer.fit_transform(
    dev_train["customer_text"]
)

# ------------------------------------------------------------
# Semantic embeddings
# ------------------------------------------------------------

X_train_embed = embedding_model.encode(
    dev_train["customer_text"].tolist(),
    normalize_embeddings=True,
    show_progress_bar=True
)

# ------------------------------------------------------------
# Combine TF-IDF + embeddings
# ------------------------------------------------------------

X_train_hybrid = hstack([
    X_train_tfidf,
    csr_matrix(X_train_embed)
])

# ------------------------------------------------------------
# Train Hybrid classifier
# ------------------------------------------------------------

hybrid_classifier = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

hybrid_classifier.fit(
    X_train_hybrid,
    dev_train["pseudo_intent"]
)

print("=" * 70)
print("HYBRID CLASSIFIER READY")
print("=" * 70)
print("Training examples:", X_train_hybrid.shape[0])
print("Features:", X_train_hybrid.shape[1])
print("Classes:", len(hybrid_classifier.classes_))

Batches:   0%|          | 0/250 [00:00<?, ?it/s]

HYBRID CLASSIFIER READY
Training examples: 8000
Features: 33062
Classes: 11


In [39]:
def classify_intent(customer_message):
    # Create TF-IDF features
    tfidf_features = tfidf_vectorizer.transform(
        [customer_message]
    )

    # Create semantic embedding
    embedding_features = embedding_model.encode(
        [customer_message],
        normalize_embeddings=True
    )

    # Combine both feature types
    hybrid_features = hstack([
        tfidf_features,
        csr_matrix(embedding_features)
    ])

    # Predict intent
    predicted_intent = hybrid_classifier.predict(
        hybrid_features
    )[0]

    # Get model confidence
    probabilities = hybrid_classifier.predict_proba(
        hybrid_features
    )[0]

    confidence = float(probabilities.max())

    return {
        "intent": predicted_intent,
        "confidence": confidence
    }


print("Intent classifier function ready!")

Intent classifier function ready!


In [40]:
test_messages = [
    "My package was supposed to arrive yesterday but I still haven't received it.",
    "Someone charged my card without my permission.",
    "I can't log into my Amazon account.",
    "My Fire Stick is not working."
]

for message in test_messages:
    result = classify_intent(message)

    print("=" * 70)
    print("CUSTOMER:", message)
    print("PREDICTED INTENT:", result["intent"])
    print("CONFIDENCE:", f"{result['confidence']:.3f}")

CUSTOMER: My package was supposed to arrive yesterday but I still haven't received it.
PREDICTED INTENT: Delivery Issue
CONFIDENCE: 0.842
CUSTOMER: Someone charged my card without my permission.
PREDICTED INTENT: Payment / Billing
CONFIDENCE: 0.957
CUSTOMER: I can't log into my Amazon account.
PREDICTED INTENT: Account / Access / Security
CONFIDENCE: 0.723
CUSTOMER: My Fire Stick is not working.
PREDICTED INTENT: Product / Device Support
CONFIDENCE: 0.721


In [41]:
def run_agent(customer_message, top_k=5):
    # 1. Classify the customer message
    classification = classify_intent(customer_message)

    intent = classification["intent"]
    confidence = classification["confidence"]

    # 2. Retrieve similar historical cases
    retrieved_cases = retrieve_cases(
        customer_message,
        top_k=top_k
    )

    # 3. Generate grounded reply
    reply = generate_reply(
        customer_message=customer_message,
        intent=intent,
        retrieved_cases=retrieved_cases
    )

    # 4. Apply escalation policy
    escalation = escalation_policy(
        customer_message=customer_message,
        intent=intent,
        retrieved_cases=retrieved_cases
    )

    return {
        "customer_message": customer_message,
        "intent": intent,
        "confidence": confidence,
        "retrieved_cases": retrieved_cases,
        "reply": reply,
        "escalation": escalation
    }


print("FINAL SupportIQ agent ready!")

FINAL SupportIQ agent ready!


In [42]:
test_messages = [
    "My package was supposed to arrive yesterday but I still haven't received it.",
    "Someone charged my card without my permission.",
    "I can't log into my Amazon account.",
    "My Fire Stick is not working.",
    "I want to speak to a human agent about my order."
]

for i, message in enumerate(test_messages, start=1):
    result = run_agent(message, top_k=5)

    print("\n" + "=" * 75)
    print(f"TEST CASE {i}")
    print("=" * 75)

    print("\nCUSTOMER:")
    print(message)

    print("\nINTENT:")
    print(result["intent"])

    print("CONFIDENCE:")
    print(f"{result['confidence']:.3f}")

    print("\nREPLY:")
    print(result["reply"])

    print("\nESCALATION:")
    print(result["escalation"]["decision"])

    print("REASON:")
    print(result["escalation"]["reason"])

    print("\nTOP RETRIEVAL:")
    for case in result["retrieved_cases"][:3]:
        print(
            f"Rank {case['rank']} | "
            f"Similarity {case['similarity']:.3f} | "
            f"{case['intent']}"
        )


TEST CASE 1

CUSTOMER:
My package was supposed to arrive yesterday but I still haven't received it.

INTENT:
Delivery Issue
CONFIDENCE:
0.842

REPLY:
I’m sorry your package hasn’t arrived yet. Please reach out to us here so we can investigate: https://t.co/hApLp

ESCALATION:
AUTO-HANDLE
REASON:
No high-risk signal or explicit human request, and relevant historical evidence was retrieved.

TOP RETRIEVAL:
Rank 1 | Similarity 0.803 | Delivery Issue
Rank 2 | Similarity 0.790 | Delivery Issue
Rank 3 | Similarity 0.745 | Delivery Issue

TEST CASE 2

CUSTOMER:
Someone charged my card without my permission.

INTENT:
Payment / Billing
CONFIDENCE:
0.957

REPLY:


ESCALATION:
AUTO-HANDLE
REASON:
No high-risk signal or explicit human request, and relevant historical evidence was retrieved.

TOP RETRIEVAL:
Rank 1 | Similarity 0.607 | Payment / Billing
Rank 2 | Similarity 0.581 | Delivery Issue
Rank 3 | Similarity 0.579 | Payment / Billing

TEST CASE 3

CUSTOMER:
I can't log into my Amazon account.

In [44]:
for message in [
    "Someone charged my card without my permission.",
    "I want to speak to a human agent about my order."
]:
    result = run_agent(message, top_k=5)

    print("=" * 70)
    print("CUSTOMER:", message)
    print("INTENT:", result["intent"])
    print("ESCALATION:", result["escalation"]["decision"])
    print("REASON:", result["escalation"]["reason"])

CUSTOMER: Someone charged my card without my permission.
INTENT: Payment / Billing
ESCALATION: ESCALATE
REASON: High-risk account, financial, legal, or safety-related issue.
CUSTOMER: I want to speak to a human agent about my order.
INTENT: Order Management
ESCALATION: ESCALATE
REASON: Customer explicitly requested human assistance.


In [45]:
def run_agent(customer_message, top_k=5):
    # 1. Classify intent
    classification = classify_intent(customer_message)

    intent = classification["intent"]
    confidence = classification["confidence"]

    # 2. Retrieve historical evidence
    retrieved_cases = retrieve_cases(
        customer_message,
        top_k=top_k
    )

    # 3. Decide escalation BEFORE generation
    escalation = escalation_policy(
        customer_message=customer_message,
        intent=intent,
        retrieved_cases=retrieved_cases
    )

    # 4. Generate reply
    reply = generate_reply(
        customer_message=customer_message,
        intent=intent,
        retrieved_cases=retrieved_cases
    )

    # 5. Return complete trace
    return {
        "customer_message": customer_message,
        "intent": intent,
        "confidence": confidence,
        "retrieved_cases": retrieved_cases,
        "reply": reply,
        "escalation_decision": escalation["decision"],
        "escalation_reason": escalation["reason"]
    }


print("Final agent trace ready!")

Final agent trace ready!


In [46]:
message = "My package was supposed to arrive yesterday but I still haven't received it."

result = run_agent(message, top_k=5)

print("=" * 70)
print("SUPPORTIQ RESULT")
print("=" * 70)

print("\nCUSTOMER:")
print(result["customer_message"])

print("\nINTENT:")
print(result["intent"])

print("\nCONFIDENCE:")
print(f"{result['confidence']:.3f}")

print("\nREPLY:")
print(result["reply"])

print("\nESCALATION:")
print(result["escalation_decision"])

print("\nESCALATION REASON:")
print(result["escalation_reason"])

print("\nRETRIEVED EVIDENCE:")
for case in result["retrieved_cases"]:
    print(
        f"Rank {case['rank']} | "
        f"Similarity {case['similarity']:.3f} | "
        f"{case['intent']}"
    )

SUPPORTIQ RESULT

CUSTOMER:
My package was supposed to arrive yesterday but I still haven't received it.

INTENT:
Delivery Issue

CONFIDENCE:
0.842

REPLY:
I’m sorry it hasn’t arrived yet! Reach out to us here: https://t.co/hApLpMlfHN

ESCALATION:
AUTO-HANDLE

ESCALATION REASON:
No escalation trigger detected and relevant historical evidence was retrieved.

RETRIEVED EVIDENCE:
Rank 1 | Similarity 0.803 | Delivery Issue
Rank 2 | Similarity 0.790 | Delivery Issue
Rank 3 | Similarity 0.745 | Delivery Issue
Rank 4 | Similarity 0.736 | Delivery Issue
Rank 5 | Similarity 0.718 | Other / Unclear


In [47]:
from pathlib import Path

agent_code = '''
def run_agent(customer_message, top_k=5):
    # 1. Classify intent
    classification = classify_intent(customer_message)

    intent = classification["intent"]
    confidence = classification["confidence"]

    # 2. Retrieve historical evidence
    retrieved_cases = retrieve_cases(
        customer_message,
        top_k=top_k
    )

    # 3. Decide escalation
    escalation = escalation_policy(
        customer_message=customer_message,
        intent=intent,
        retrieved_cases=retrieved_cases
    )

    # 4. Generate grounded reply
    reply = generate_reply(
        customer_message=customer_message,
        intent=intent,
        retrieved_cases=retrieved_cases
    )

    # 5. Return complete trace
    return {
        "customer_message": customer_message,
        "intent": intent,
        "confidence": confidence,
        "retrieved_cases": retrieved_cases,
        "reply": reply,
        "escalation_decision": escalation["decision"],
        "escalation_reason": escalation["reason"]
    }
'''

path = Path("../src/pipeline/agent.py")
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(agent_code.strip(), encoding="utf-8")

print("Saved:", path)

Saved: ..\src\pipeline\agent.py


In [48]:
from pathlib import Path

agent_code = '''
def run_agent(customer_message, top_k=5):
    # 1. Classify intent
    classification = classify_intent(customer_message)

    intent = classification["intent"]
    confidence = classification["confidence"]

    # 2. Retrieve historical evidence
    retrieved_cases = retrieve_cases(
        customer_message,
        top_k=top_k
    )

    # 3. Decide escalation
    escalation = escalation_policy(
        customer_message=customer_message,
        intent=intent,
        retrieved_cases=retrieved_cases
    )

    # 4. Generate grounded reply
    reply = generate_reply(
        customer_message=customer_message,
        intent=intent,
        retrieved_cases=retrieved_cases
    )

    # 5. Return complete trace
    return {
        "customer_message": customer_message,
        "intent": intent,
        "confidence": confidence,
        "retrieved_cases": retrieved_cases,
        "reply": reply,
        "escalation_decision": escalation["decision"],
        "escalation_reason": escalation["reason"]
    }
'''

path = Path("../src/pipeline/agent.py")
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(agent_code.strip(), encoding="utf-8")

print("Saved:", path)

Saved: ..\src\pipeline\agent.py


In [49]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

# Prepare Golden Set customer text
golden_text = golden["conversation"].apply(extract_customer_text)

# Predict each Golden example
golden_predictions = []

for text in golden_text:
    result = classify_intent(text)
    golden_predictions.append(result["intent"])

# True labels
golden_true = golden["intent"].tolist()

# Metrics
accuracy = accuracy_score(golden_true, golden_predictions)
macro_precision = precision_score(
    golden_true,
    golden_predictions,
    average="macro",
    zero_division=0
)
macro_recall = recall_score(
    golden_true,
    golden_predictions,
    average="macro",
    zero_division=0
)
macro_f1 = f1_score(
    golden_true,
    golden_predictions,
    average="macro",
    zero_division=0
)

print("=" * 70)
print("SUPPORTIQ — GOLDEN SET CLASSIFICATION EVALUATION")
print("=" * 70)

print(f"Examples          : {len(golden_true)}")
print(f"Accuracy          : {accuracy:.3f} ({accuracy:.2%})")
print(f"Macro Precision   : {macro_precision:.3f}")
print(f"Macro Recall      : {macro_recall:.3f}")
print(f"Macro F1          : {macro_f1:.3f}")

print("\nPER-INTENT RESULTS")
print("-" * 70)

print(
    classification_report(
        golden_true,
        golden_predictions,
        zero_division=0
    )
)

SUPPORTIQ — GOLDEN SET CLASSIFICATION EVALUATION
Examples          : 200
Accuracy          : 0.540 (54.00%)
Macro Precision   : 0.491
Macro Recall      : 0.514
Macro F1          : 0.484

PER-INTENT RESULTS
----------------------------------------------------------------------
                                precision    recall  f1-score   support

   Account / Access / Security       0.38      0.38      0.38         8
Damaged / Wrong / Missing Item       0.61      0.52      0.56        21
                Delivery Issue       0.80      0.59      0.68        68
               Digital Content       0.73      0.73      0.73        15
              Order Management       0.21      0.38      0.27         8
               Other / Unclear       0.31      0.52      0.39        23
             Payment / Billing       0.46      0.55      0.50        11
              Prime Membership       0.62      0.38      0.48        13
      Product / Device Support       0.40      0.75      0.52         8
Pr

In [50]:
# ------------------------------------------------------------
# Retrieval evaluation using intent-match as proxy relevance
# ------------------------------------------------------------

retrieval_results = []

for i, row in golden.iterrows():

    customer_text = row["customer_text"]
    true_intent = row["intent"]

    results = retrieve_cases(
        customer_text,
        top_k=10
    )

    retrieved_intents = [
        case["intent"]
        for case in results
    ]

    retrieval_results.append({
        "root_tweet_id": row["root_tweet_id"],
        "true_intent": true_intent,
        "retrieved_intents": retrieved_intents
    })

print("Retrieval evaluation examples:", len(retrieval_results))

Retrieval evaluation examples: 200


In [51]:
import numpy as np

def calculate_recall_at_k(results, k):
    hits = 0

    for result in results:
        true_intent = result["true_intent"]
        top_k_intents = result["retrieved_intents"][:k]

        if true_intent in top_k_intents:
            hits += 1

    return hits / len(results)


def calculate_mrr(results):
    reciprocal_ranks = []

    for result in results:
        true_intent = result["true_intent"]
        retrieved_intents = result["retrieved_intents"]

        rank = None

        for i, intent in enumerate(retrieved_intents, start=1):
            if intent == true_intent:
                rank = i
                break

        reciprocal_ranks.append(
            1 / rank if rank is not None else 0
        )

    return np.mean(reciprocal_ranks)


recall_1 = calculate_recall_at_k(retrieval_results, 1)
recall_3 = calculate_recall_at_k(retrieval_results, 3)
recall_5 = calculate_recall_at_k(retrieval_results, 5)
recall_10 = calculate_recall_at_k(retrieval_results, 10)

mrr = calculate_mrr(retrieval_results)

print("=" * 70)
print("SUPPORTIQ — RETRIEVAL EVALUATION")
print("=" * 70)

print(f"Examples  : {len(retrieval_results)}")
print(f"Recall@1  : {recall_1:.3f} ({recall_1:.2%})")
print(f"Recall@3  : {recall_3:.3f} ({recall_3:.2%})")
print(f"Recall@5  : {recall_5:.3f} ({recall_5:.2%})")
print(f"Recall@10 : {recall_10:.3f} ({recall_10:.2%})")
print(f"MRR       : {mrr:.3f}")

SUPPORTIQ — RETRIEVAL EVALUATION
Examples  : 200
Recall@1  : 0.375 (37.50%)
Recall@3  : 0.610 (61.00%)
Recall@5  : 0.700 (70.00%)
Recall@10 : 0.790 (79.00%)
MRR       : 0.515


In [52]:
from pathlib import Path
import json

results_dir = Path("../results")
results_dir.mkdir(parents=True, exist_ok=True)

retrieval_metrics = {
    "evaluation_examples": len(retrieval_results),
    "relevance_definition": "Retrieved case is considered relevant when its historical pseudo-intent matches the Golden Set human intent.",
    "recall_at_1": round(recall_1, 4),
    "recall_at_3": round(recall_3, 4),
    "recall_at_5": round(recall_5, 4),
    "recall_at_10": round(recall_10, 4),
    "mrr": round(float(mrr), 4)
}

with open(
    results_dir / "retrieval_metrics.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(retrieval_metrics, f, indent=2)

print("Saved:", results_dir / "retrieval_metrics.json")
print(json.dumps(retrieval_metrics, indent=2))

Saved: ..\results\retrieval_metrics.json
{
  "evaluation_examples": 200,
  "relevance_definition": "Retrieved case is considered relevant when its historical pseudo-intent matches the Golden Set human intent.",
  "recall_at_1": 0.375,
  "recall_at_3": 0.61,
  "recall_at_5": 0.7,
  "recall_at_10": 0.79,
  "mrr": 0.515
}


In [53]:
def reference_escalation_label(text):
    text = str(text).lower()

    high_risk_keywords = [
        "hacked",
        "hack",
        "account stolen",
        "unauthorized",
        "unauthorised",
        "without my permission",
        "fraud",
        "scam",
        "stolen",
        "chargeback",
        "lawsuit",
        "legal",
        "police",
        "danger",
        "injured",
        "unsafe"
    ]

    human_request_keywords = [
        "human",
        "agent",
        "representative",
        "real person",
        "speak to someone",
        "talk to someone",
        "speak with someone"
    ]

    if any(keyword in text for keyword in high_risk_keywords):
        return "ESCALATE"

    if any(keyword in text for keyword in human_request_keywords):
        return "ESCALATE"

    return "AUTO-HANDLE"


# Create reference labels for Golden Set
golden["reference_escalation"] = golden["customer_text"].apply(
    reference_escalation_label
)

print("=" * 70)
print("REFERENCE ESCALATION LABELS")
print("=" * 70)

print(
    golden["reference_escalation"]
    .value_counts()
)

REFERENCE ESCALATION LABELS
reference_escalation
AUTO-HANDLE    195
ESCALATE         5
Name: count, dtype: int64


In [54]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

actual_escalations = []
reference_escalations = []

for _, row in golden.iterrows():

    result = run_agent(
        row["customer_text"],
        top_k=5
    )

    actual_escalations.append(
        result["escalation_decision"]
    )

    reference_escalations.append(
        row["reference_escalation"]
    )


# Convert to binary
y_true = [
    1 if x == "ESCALATE" else 0
    for x in reference_escalations
]

y_pred = [
    1 if x == "ESCALATE" else 0
    for x in actual_escalations
]


precision = precision_score(
    y_true,
    y_pred,
    zero_division=0
)

recall = recall_score(
    y_true,
    y_pred,
    zero_division=0
)

f1 = f1_score(
    y_true,
    y_pred,
    zero_division=0
)

cm = confusion_matrix(y_true, y_pred)

# Missed escalation = actual escalation required,
# but system chose AUTO-HANDLE
missed_escalations = sum(
    true == 1 and pred == 0
    for true, pred in zip(y_true, y_pred)
)

missed_escalation_rate = (
    missed_escalations / sum(y_true)
    if sum(y_true) > 0
    else 0
)


print("=" * 70)
print("SUPPORTIQ — ESCALATION EVALUATION")
print("=" * 70)

print(f"Examples                : {len(y_true)}")
print(f"Reference escalations   : {sum(y_true)}")
print(f"System escalations      : {sum(y_pred)}")

print(f"\nPrecision               : {precision:.3f}")
print(f"Recall                  : {recall:.3f}")
print(f"F1                      : {f1:.3f}")
print(f"Missed escalation rate  : {missed_escalation_rate:.3f}")

print("\nConfusion Matrix")
print("[[TN FP]")
print(" [FN TP]]")
print(cm)

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-20b` in organization `org_01m2n8dz4vepqbmdrgm8sz56kk` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 199263, Requested 1759. Please try again in 7m21.504s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

In [55]:
# Check how much of the Golden set we already generated, if anything

from pathlib import Path
import pandas as pd

output_path = Path("../evaluation/golden_agent_outputs.csv")

if output_path.exists():
    existing = pd.read_csv(output_path)
    print("Existing outputs:", len(existing))
    print(existing.columns.tolist())
else:
    print("No saved agent outputs yet.")

No saved agent outputs yet.


In [56]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

actual_escalations = []
reference_escalations = []

for _, row in golden.iterrows():

    # Retrieve historical cases only
    retrieved_cases = retrieve_cases(
        row["customer_text"],
        top_k=5
    )

    # Evaluate escalation policy directly
    result = escalation_policy(
        customer_message=row["customer_text"],
        intent=row["intent"],
        retrieved_cases=retrieved_cases
    )

    actual_escalations.append(result["decision"])
    reference_escalations.append(row["reference_escalation"])


# Convert to binary
y_true = [
    1 if x == "ESCALATE" else 0
    for x in reference_escalations
]

y_pred = [
    1 if x == "ESCALATE" else 0
    for x in actual_escalations
]


# Metrics
precision = precision_score(
    y_true,
    y_pred,
    zero_division=0
)

recall = recall_score(
    y_true,
    y_pred,
    zero_division=0
)

f1 = f1_score(
    y_true,
    y_pred,
    zero_division=0
)

cm = confusion_matrix(y_true, y_pred)


# Missed escalation:
# reference says ESCALATE,
# system says AUTO-HANDLE
missed_escalations = sum(
    true == 1 and pred == 0
    for true, pred in zip(y_true, y_pred)
)

missed_escalation_rate = (
    missed_escalations / sum(y_true)
    if sum(y_true) > 0
    else 0
)


print("=" * 70)
print("SUPPORTIQ — ESCALATION POLICY EVALUATION")
print("=" * 70)

print(f"Examples                : {len(y_true)}")
print(f"Reference escalations   : {sum(y_true)}")
print(f"System escalations      : {sum(y_pred)}")

print(f"\nPrecision               : {precision:.3f}")
print(f"Recall                  : {recall:.3f}")
print(f"F1                      : {f1:.3f}")
print(f"Missed escalation rate  : {missed_escalation_rate:.3f}")

print("\nConfusion Matrix")
print("[[TN FP]")
print(" [FN TP]]")
print(cm)

SUPPORTIQ — ESCALATION POLICY EVALUATION
Examples                : 200
Reference escalations   : 5
System escalations      : 9

Precision               : 0.556
Recall                  : 1.000
F1                      : 0.714
Missed escalation rate  : 0.000

Confusion Matrix
[[TN FP]
 [FN TP]]
[[191   4]
 [  0   5]]


In [57]:
import pandas as pd
from pathlib import Path

# Load the final Golden set
golden = pd.read_csv("../data/golden/golden_set.csv")

# Fixed 50-example subset for reply evaluation
eval_subset = golden.sample(
    n=50,
    random_state=2026
).copy()

# Save it so we never accidentally change the evaluation examples
output_path = Path("../evaluation/eval_subset_50.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

eval_subset.to_csv(output_path, index=False)

print("Evaluation examples:", len(eval_subset))
print("Saved to:", output_path)

print("\nIntent distribution:")
print(eval_subset["intent"].value_counts())

Evaluation examples: 50
Saved to: ..\evaluation\eval_subset_50.csv

Intent distribution:
intent
Delivery Issue                    17
Damaged / Wrong / Missing Item     6
Other / Unclear                    5
Return / Refund                    5
Account / Access / Security        4
Product / Device Support           4
Order Management                   3
Payment / Billing                  3
Digital Content                    2
Prime Membership                   1
Name: count, dtype: int64


In [58]:
import pandas as pd
from pathlib import Path
import time

# Load fixed evaluation subset
eval_subset = pd.read_csv("../evaluation/eval_subset_50.csv")

output_path = Path("../evaluation/golden_agent_outputs.csv")

results = []

for i, row in eval_subset.iterrows():

    print(f"Processing {i + 1}/{len(eval_subset)}...")

    result = run_agent(
        row["customer_text"],
        top_k=5
    )

    results.append({
        "root_tweet_id": row["root_tweet_id"],
        "customer_text": row["customer_text"],
        "golden_intent": row["intent"],
        "predicted_intent": result["intent"],
        "confidence": result["confidence"],
        "reply": result["reply"],
        "escalation_decision": result["escalation_decision"],
        "escalation_reason": result["escalation_reason"]
    })

    # Small pause between API calls
    time.sleep(1)

# Save all results
agent_outputs = pd.DataFrame(results)

output_path.parent.mkdir(parents=True, exist_ok=True)
agent_outputs.to_csv(output_path, index=False)

print("\n" + "=" * 60)
print("AGENT EVALUATION COMPLETE")
print("=" * 60)

print("Examples:", len(agent_outputs))
print("Saved:", output_path)

print("\nEscalation distribution:")
print(agent_outputs["escalation_decision"].value_counts())

print("\nEmpty replies:",
      agent_outputs["reply"].fillna("").str.strip().eq("").sum())

Processing 1/50...


KeyError: 'customer_text'

In [59]:
import pandas as pd

# Load the frozen 50-example evaluation set
eval_subset = pd.read_csv("../evaluation/eval_subset_50.csv")

# Extract customer messages from each conversation
def extract_customer_text(conversation):
    lines = str(conversation).splitlines()

    customer_messages = []

    for line in lines:
        if line.startswith("CUSTOMER:"):
            message = line.replace("CUSTOMER:", "", 1).strip()
            customer_messages.append(message)

    return " ".join(customer_messages)


# Create customer_text
eval_subset["customer_text"] = eval_subset["conversation"].apply(
    extract_customer_text
)

# Save updated file
eval_subset.to_csv(
    "../evaluation/eval_subset_50.csv",
    index=False
)

print("Updated evaluation set.")
print("Examples:", len(eval_subset))
print("Columns:", eval_subset.columns.tolist())

print("\nFirst customer message:")
print(eval_subset.iloc[0]["customer_text"])

Updated evaluation set.
Examples: 50
Columns: ['root_tweet_id', 'thread_size', 'conversation', 'intent', 'annotation_notes', 'customer_text']

First customer message:
@1384 @115850


In [63]:
import pandas as pd
from pathlib import Path
import time

# Load the fixed 50-example evaluation set
eval_subset = pd.read_csv("../evaluation/eval_subset_50.csv")

output_path = Path("../evaluation/golden_agent_outputs.csv")

results = []

for i, row in eval_subset.iterrows():

    print(f"Processing {i + 1}/{len(eval_subset)}...")

    result = run_agent(
        row["customer_text"],
        top_k=5
    )

    results.append({
        "root_tweet_id": row["root_tweet_id"],
        "customer_text": row["customer_text"],
        "golden_intent": row["intent"],
        "predicted_intent": result["intent"],
        "confidence": result["confidence"],
        "reply": result["reply"],
        "escalation_decision": result["escalation_decision"],
        "escalation_reason": result["escalation_reason"]
    })

    # Avoid sending requests too rapidly
    time.sleep(1)

# Save results
agent_outputs = pd.DataFrame(results)

output_path.parent.mkdir(parents=True, exist_ok=True)
agent_outputs.to_csv(output_path, index=False)

print("\n" + "=" * 60)
print("AGENT EVALUATION COMPLETE")
print("=" * 60)

print("Examples:", len(agent_outputs))
print("Saved:", output_path)

print("\nEscalation distribution:")
print(agent_outputs["escalation_decision"].value_counts())

print(
    "\nEmpty replies:",
    agent_outputs["reply"].fillna("").str.strip().eq("").sum()
)

Processing 1/50...


AuthenticationError: Error code: 401 - {'error': {'message': 'Invalid API Key', 'type': 'invalid_request_error', 'code': 'invalid_api_key'}}

In [64]:
import os
from dotenv import load_dotenv

env_path = r"K:\projects\sales agent hiver\.env"

loaded = load_dotenv(env_path, override=True)

key = os.getenv("GROQ_API_KEY")
model = os.getenv("GROQ_MODEL")

print("ENV loaded:", loaded)
print("API key exists:", bool(key))
print("API key length:", len(key) if key else 0)
print("API key prefix:", key[:8] + "..." if key else "NONE")
print("Model:", model)

ENV loaded: True
API key exists: True
API key length: 56
API key prefix: gsk_pys6...
Model: openai/gpt-oss-20b


In [65]:
import os
from dotenv import load_dotenv
from openai import OpenAI

env_path = r"K:\projects\sales agent hiver\.env"
load_dotenv(env_path, override=True)

client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

response = client.chat.completions.create(
    model=os.getenv("GROQ_MODEL", "openai/gpt-oss-20b"),
    messages=[
        {
            "role": "user",
            "content": "Reply with exactly: GROQ WORKS"
        }
    ],
    max_tokens=10
)

print(response.choices[0].message.content)

In [66]:
print("Response object:")
print(response)

print("\nChoices:")
print(response.choices)

if response.choices:
    print("\nMessage:")
    print(response.choices[0].message)

    print("\nContent:")
    print(repr(response.choices[0].message.content))

    print("\nFinish reason:")
    print(response.choices[0].finish_reason)

Response object:
ChatCompletion(id='chatcmpl-3c9b952c-9a05-45b5-9e12-f38b746b65f2', choices=[Choice(finish_reason='length', index=0, logprobs=None, message=ChatCompletionMessage(content='', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, reasoning='The user says: "Reply with'))], created=1789574386, model='openai/gpt-oss-20b', object='chat.completion', metadata=None, moderation=None, service_tier='on_demand', system_fingerprint='fp_565badff47', usage=CompletionUsage(completion_tokens=10, prompt_tokens=79, total_tokens=89, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=8, rejected_prediction_tokens=None, text_tokens=None), prompt_tokens_details=None, queue_time=0.314474875, prompt_time=0.003811571, completion_time=0.010239186, total_time=0.014050757), usage_breakdown=None, x_groq={'id': 'req_01m2nf443wef2apbx2033h68cy', 'seed': 1423018180})

Choices:
[Choice(finish_

In [67]:
response = client.chat.completions.create(
    model=os.getenv("GROQ_MODEL", "openai/gpt-oss-20b"),
    messages=[
        {
            "role": "user",
            "content": "Reply with exactly: GROQ WORKS"
        }
    ],
    max_tokens=100
)

print("Content:", repr(response.choices[0].message.content))
print("Finish reason:", response.choices[0].finish_reason)
print("Reasoning:", repr(response.choices[0].message.reasoning))

Content: 'GROQ WORKS'
Finish reason: stop
Reasoning: 'The user says: "Reply with exactly: GROQ WORKS". So I must reply exactly that string. No quotes. No additional text.'


In [68]:
import pandas as pd
from pathlib import Path
import time

# Load fixed 50-example evaluation set
eval_subset = pd.read_csv("../evaluation/eval_subset_50.csv")

output_path = Path("../evaluation/golden_agent_outputs.csv")

# Load previous progress if it exists
if output_path.exists():
    agent_outputs = pd.read_csv(output_path)
    completed_ids = set(agent_outputs["root_tweet_id"].astype(str))
    results = agent_outputs.to_dict("records")

    print(f"Resuming: {len(completed_ids)} already completed")
else:
    completed_ids = set()
    results = []
    print("Starting from 0")


for _, row in eval_subset.iterrows():

    root_id = str(row["root_tweet_id"])

    # Skip already completed examples
    if root_id in completed_ids:
        continue

    print(
        f"Processing {len(results) + 1}/{len(eval_subset)}..."
    )

    try:
        result = run_agent(
            row["customer_text"],
            top_k=5
        )

        results.append({
            "root_tweet_id": row["root_tweet_id"],
            "customer_text": row["customer_text"],
            "golden_intent": row["intent"],
            "predicted_intent": result["intent"],
            "confidence": result["confidence"],
            "reply": result["reply"],
            "escalation_decision": result["escalation_decision"],
            "escalation_reason": result["escalation_reason"]
        })

        # Save after EVERY successful example
        pd.DataFrame(results).to_csv(
            output_path,
            index=False
        )

        time.sleep(1)

    except Exception as e:

        # Save progress before stopping
        pd.DataFrame(results).to_csv(
            output_path,
            index=False
        )

        print("\nERROR:")
        print(type(e).__name__, e)

        print(
            f"\nProgress saved: {len(results)} examples"
        )

        break


# Final summary
agent_outputs = pd.DataFrame(results)

print("\n" + "=" * 60)
print("GENERATION PROGRESS")
print("=" * 60)

print("Completed:", len(agent_outputs))
print("Remaining:", len(eval_subset) - len(agent_outputs))
print("Saved:", output_path)

if len(agent_outputs) > 0:
    print("\nEscalation distribution:")
    print(agent_outputs["escalation_decision"].value_counts())

    print(
        "\nEmpty replies:",
        agent_outputs["reply"].fillna("").str.strip().eq("").sum()
    )

Starting from 0
Processing 1/50...
Processing 2/50...
Processing 3/50...
Processing 4/50...
Processing 5/50...
Processing 6/50...
Processing 7/50...
Processing 8/50...
Processing 9/50...
Processing 10/50...
Processing 11/50...
Processing 12/50...
Processing 13/50...
Processing 14/50...
Processing 15/50...
Processing 16/50...
Processing 17/50...
Processing 18/50...
Processing 19/50...
Processing 20/50...
Processing 21/50...
Processing 22/50...
Processing 23/50...
Processing 24/50...
Processing 25/50...
Processing 26/50...
Processing 27/50...
Processing 28/50...
Processing 29/50...
Processing 30/50...
Processing 31/50...
Processing 32/50...
Processing 33/50...
Processing 34/50...
Processing 35/50...
Processing 36/50...
Processing 37/50...
Processing 38/50...
Processing 39/50...
Processing 40/50...
Processing 41/50...
Processing 42/50...
Processing 43/50...
Processing 44/50...
Processing 45/50...
Processing 46/50...
Processing 47/50...
Processing 48/50...
Processing 49/50...
Processing 50

In [69]:
import pandas as pd

agent_outputs = pd.read_csv(
    "../evaluation/golden_agent_outputs.csv"
)

empty = agent_outputs[
    agent_outputs["reply"].fillna("").str.strip() == ""
].copy()

print("Empty replies:", len(empty))

print("\nEscalation distribution among empty replies:")
print(empty["escalation_decision"].value_counts())

print("\nEmpty reply examples:")
print(
    empty[
        [
            "root_tweet_id",
            "customer_text",
            "predicted_intent",
            "confidence",
            "escalation_decision",
            "escalation_reason"
        ]
    ].to_string(index=False)
)

Empty replies: 23

Escalation distribution among empty replies:
escalation_decision
AUTO-HANDLE    20
ESCALATE        3
Name: count, dtype: int64

Empty reply examples:
 root_tweet_id                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                            customer_text               predicted_intent  confidence escalation_decision                                                              escalation_reason
       2396389                                                                                                                                           

In [70]:
import pandas as pd
from pathlib import Path
import time

path = "../evaluation/golden_agent_outputs.csv"

# Load the existing 50 results
agent_outputs = pd.read_csv(path)

# Find exactly the 23 empty replies
empty_mask = (
    agent_outputs["reply"]
    .fillna("")
    .str.strip()
    .eq("")
)

empty_rows = agent_outputs[empty_mask].copy()

print("Empty replies found:", len(empty_rows))

results = []

for i, (_, row) in enumerate(empty_rows.iterrows(), start=1):

    print(f"Regenerating {i}/{len(empty_rows)}...")

    result = run_agent(
        row["customer_text"],
        top_k=5
    )

    results.append({
        "root_tweet_id": row["root_tweet_id"],
        "reply": result["reply"],
        "predicted_intent": result["intent"],
        "confidence": result["confidence"],
        "escalation_decision": result["escalation_decision"],
        "escalation_reason": result["escalation_reason"]
    })

    time.sleep(1)


# Replace only the 23 empty replies
for result in results:

    mask = (
        agent_outputs["root_tweet_id"].astype(str)
        == str(result["root_tweet_id"])
    )

    agent_outputs.loc[mask, "reply"] = result["reply"]
    agent_outputs.loc[mask, "predicted_intent"] = result["predicted_intent"]
    agent_outputs.loc[mask, "confidence"] = result["confidence"]
    agent_outputs.loc[mask, "escalation_decision"] = result["escalation_decision"]
    agent_outputs.loc[mask, "escalation_reason"] = result["escalation_reason"]


# Save
agent_outputs.to_csv(path, index=False)

# Check remaining empty replies
remaining_empty = (
    agent_outputs["reply"]
    .fillna("")
    .str.strip()
    .eq("")
    .sum()
)

print("\n" + "=" * 60)
print("REGENERATION COMPLETE")
print("=" * 60)
print("Regenerated:", len(results))
print("Remaining empty replies:", remaining_empty)
print("Saved:", path)

Empty replies found: 23
Regenerating 1/23...
Regenerating 2/23...
Regenerating 3/23...
Regenerating 4/23...
Regenerating 5/23...
Regenerating 6/23...
Regenerating 7/23...
Regenerating 8/23...
Regenerating 9/23...
Regenerating 10/23...
Regenerating 11/23...
Regenerating 12/23...
Regenerating 13/23...
Regenerating 14/23...
Regenerating 15/23...
Regenerating 16/23...
Regenerating 17/23...
Regenerating 18/23...
Regenerating 19/23...
Regenerating 20/23...
Regenerating 21/23...
Regenerating 22/23...
Regenerating 23/23...

REGENERATION COMPLETE
Regenerated: 23
Remaining empty replies: 12
Saved: ../evaluation/golden_agent_outputs.csv


In [71]:
import pandas as pd

agent_outputs = pd.read_csv(
    "../evaluation/golden_agent_outputs.csv"
)

empty_mask = (
    agent_outputs["reply"]
    .fillna("")
    .str.strip()
    .eq("")
)

empty = agent_outputs[empty_mask].copy()

print("Remaining empty replies:", len(empty))

print("\nEscalation distribution:")
print(empty["escalation_decision"].value_counts())

print("\nRemaining empty examples:")
print(
    empty[
        [
            "root_tweet_id",
            "customer_text",
            "predicted_intent",
            "confidence",
            "escalation_decision",
            "escalation_reason"
        ]
    ].to_string(index=False)
)

Remaining empty replies: 12

Escalation distribution:
escalation_decision
AUTO-HANDLE    11
ESCALATE        1
Name: count, dtype: int64

Remaining empty examples:
 root_tweet_id                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                            customer_text         predicted_intent  confidence escalation_decision                                                              escalation_reason
       2396389                                                                                                                                                       

In [72]:
import pandas as pd

agent_outputs = pd.read_csv(
    "../evaluation/golden_agent_outputs.csv"
)

non_empty = agent_outputs[
    agent_outputs["reply"].fillna("").str.strip() != ""
].copy()

print("Non-empty replies:", len(non_empty))
print("Empty replies:", len(agent_outputs) - len(non_empty))

print("\nSample generated replies:\n")

for _, row in non_empty.head(10).iterrows():
    print("=" * 70)
    print("CUSTOMER:", row["customer_text"])
    print("INTENT:", row["predicted_intent"])
    print("ESCALATION:", row["escalation_decision"])
    print("REPLY:", row["reply"])

Non-empty replies: 38
Empty replies: 12

Sample generated replies:

CUSTOMER: @1384 @115850
INTENT: Product / Device Support
ESCALATION: AUTO-HANDLE
REPLY: Hi! Thanks for reaching out. Could you let us know what you need help with?
CUSTOMER: When you peel open your @115821 box and there's 50+ giant ants eating something... #nasty
INTENT: Product / Device Support
ESCALATION: ESCALATE
REPLY: I’m really sorry to hear about this.  
Could you please share the order details and any photos of the packaging here: https://t.co/GIJyeYqKE0? We’ll look into it right away.
CUSTOMER: @115821 yo why do my book orders keep getting sent with no kind of padding or protection in the boxes
INTENT: Damaged / Wrong / Missing Item
ESCALATION: AUTO-HANDLE
REPLY: I’m sorry to hear that your books are arriving without proper protection. Were any of the books damaged or missing when you opened the package? If so, please let me know which items were affected. You can also share your feedback about the packaging h

In [73]:
import pandas as pd
import json

# Load the 50 generated outputs
agent_outputs = pd.read_csv(
    "../evaluation/golden_agent_outputs.csv"
)

# Keep only replies that actually contain generated text
judge_data = agent_outputs[
    agent_outputs["reply"].fillna("").str.strip() != ""
].copy()

# Retrieve the historical evidence again
retrieved_evidence = []

for _, row in judge_data.iterrows():

    cases = retrieve_cases(
        row["customer_text"],
        top_k=5
    )

    # Keep only the information the judge needs
    evidence = []

    for case in cases:
        evidence.append({
            "rank": case["rank"],
            "similarity": case["similarity"],
            "intent": case["intent"],
            "conversation": case["conversation"]
        })

    retrieved_evidence.append(
        json.dumps(evidence, ensure_ascii=False)
    )

judge_data["retrieved_evidence"] = retrieved_evidence

# Save judge input
judge_data.to_csv(
    "../evaluation/reply_judge_input.csv",
    index=False
)

print("Judge examples:", len(judge_data))
print("Saved: ../evaluation/reply_judge_input.csv")

print("\nColumns:")
print(judge_data.columns.tolist())

Judge examples: 38
Saved: ../evaluation/reply_judge_input.csv

Columns:
['root_tweet_id', 'customer_text', 'golden_intent', 'predicted_intent', 'confidence', 'reply', 'escalation_decision', 'escalation_reason', 'retrieved_evidence']


In [76]:
from pathlib import Path

path = Path("../evaluation/reply_judge_results.csv")

if path.exists():
    path.unlink()
    print("Old judge results deleted.")
else:
    print("No judge results found.")

print("Ready for fresh judge run.")

Old judge results deleted.
Ready for fresh judge run.


In [78]:
models = client.models.list()

for model in models.data:
    print(model.id)

qwen/qwen3.8-27b
whisper-large-v3-turbo
groq/compound-mini
canopylabs/orpheus-arabic-saudi
openai/gpt-oss-120b
meta-llama/llama-prompt-guard-2-22m
whisper-large-v3
meta-llama/llama-prompt-guard-2-86m
openai/gpt-oss-safeguard-20b
openai/gpt-oss-20b
allam-2-7b
groq/compound
canopylabs/orpheus-v1-english


In [79]:
judge_model = "qwen/qwen3.8-27b"

response = client.chat.completions.create(
    model=judge_model,
    messages=[
        {
            "role": "user",
            "content": """
Score this support reply.

Customer: My package is late.

Reply: Sorry your package is late. Please contact support with your order number.

Return ONLY this JSON:
{"correctness":4,"groundedness":4,"relevance":5,"tone":5,"actionability":4}
"""
        }
    ],
    max_tokens=200,
    temperature=0
)

print("Content:")
print(repr(response.choices[0].message.content))

print("\nFinish reason:")
print(response.choices[0].finish_reason)

Content:
'{"correctness":4,"groundedness":4,"relevance":5,"tone":5,"actionability":4}'

Finish reason:
stop


In [81]:
import pandas as pd
import json
import time
import os
from pathlib import Path

input_path = "../evaluation/reply_judge_input.csv"
output_path = "../evaluation/reply_judge_results.csv"

judge_data = pd.read_csv(input_path)

# Safely load previous results
results = []
completed_ids = set()

if Path(output_path).exists():

    try:
        existing = pd.read_csv(output_path)

        if len(existing) > 0:
            results = existing.to_dict("records")
            completed_ids = set(
                existing["root_tweet_id"].astype(str)
            )

            print("Already judged:", len(completed_ids))
        else:
            print("Existing judge file is empty. Starting from 0.")

    except pd.errors.EmptyDataError:
        print("Existing judge file is empty. Starting from 0.")

else:
    print("No previous judge results. Starting from 0.")


judge_model = "qwen/qwen3.8-27b"


def judge_reply(row):

    prompt = f"""
Evaluate this AI customer-support reply.

CUSTOMER:
{row["customer_text"]}

EXPECTED INTENT:
{row["golden_intent"]}

AI REPLY:
{row["reply"]}

HISTORICAL SUPPORT EVIDENCE:
{row["retrieved_evidence"]}

Score each criterion from 1 to 5:

correctness: Does the reply address the customer's issue?

groundedness: Is the reply supported by the historical evidence?
Penalize invented facts, policies, guarantees, actions, refunds, or unsupported links.

relevance: Is the reply focused on the customer's problem?

tone: Is it polite and professional?

actionability: Does it provide a useful next step?

Return ONLY this JSON object:

{{
  "correctness": 1,
  "groundedness": 1,
  "relevance": 1,
  "tone": 1,
  "actionability": 1
}}
"""

    response = client.chat.completions.create(
        model=judge_model,
        messages=[
            {
                "role": "system",
                "content": "Return only valid JSON. No Markdown. No explanation."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        max_tokens=200,
        temperature=0
    )

    content = response.choices[0].message.content.strip()

    start = content.find("{")
    end = content.rfind("}")

    if start == -1 or end == -1:
        raise ValueError(
            f"No JSON found: {repr(content[:300])}"
        )

    return json.loads(
        content[start:end + 1]
    )


for _, row in judge_data.iterrows():

    root_id = str(row["root_tweet_id"])

    if root_id in completed_ids:
        continue

    print(
        f"Judging {len(results) + 1}/{len(judge_data)}..."
    )

    try:

        scores = judge_reply(row)

        results.append({
            "root_tweet_id": row["root_tweet_id"],
            "correctness": scores["correctness"],
            "groundedness": scores["groundedness"],
            "relevance": scores["relevance"],
            "tone": scores["tone"],
            "actionability": scores["actionability"]
        })

        completed_ids.add(root_id)

        # Save after every successful judgment
        pd.DataFrame(results).to_csv(
            output_path,
            index=False
        )

        time.sleep(1)

    except Exception as e:

        pd.DataFrame(results).to_csv(
            output_path,
            index=False
        )

        print("\nERROR:")
        print(type(e).__name__, e)

        print(
            "Model output:",
            repr(
                content[:500]
                if "content" in locals()
                else ""
            )
        )

        print(
            f"Progress saved: {len(results)}"
        )

        break


judge_results = pd.DataFrame(results)

print("\n" + "=" * 60)
print("LLM JUDGE PROGRESS")
print("=" * 60)

print("Judged:", len(judge_results))
print(
    "Remaining:",
    len(judge_data) - len(judge_results)
)

print("Saved:", output_path)

if len(judge_results) > 0:

    print("\nAverage scores:")

    for column in [
        "correctness",
        "groundedness",
        "relevance",
        "tone",
        "actionability"
    ]:

        print(
            f"{column:15s}: "
            f"{judge_results[column].mean():.2f}/5"
        )

Existing judge file is empty. Starting from 0.
Judging 1/38...
Judging 2/38...
Judging 3/38...
Judging 4/38...
Judging 5/38...
Judging 6/38...
Judging 7/38...
Judging 8/38...
Judging 9/38...
Judging 10/38...
Judging 11/38...
Judging 12/38...
Judging 13/38...
Judging 14/38...
Judging 15/38...
Judging 16/38...
Judging 17/38...
Judging 18/38...
Judging 19/38...
Judging 20/38...
Judging 21/38...
Judging 22/38...
Judging 23/38...
Judging 24/38...
Judging 25/38...
Judging 26/38...
Judging 27/38...
Judging 28/38...
Judging 29/38...
Judging 30/38...
Judging 31/38...
Judging 32/38...
Judging 33/38...
Judging 34/38...
Judging 35/38...
Judging 36/38...
Judging 37/38...
Judging 38/38...

LLM JUDGE PROGRESS
Judged: 38
Remaining: 0
Saved: ../evaluation/reply_judge_results.csv

Average scores:
correctness    : 3.21/5
groundedness   : 4.00/5
relevance      : 3.66/5
tone           : 4.29/5
actionability  : 3.03/5


In [82]:
import pandas as pd
from pathlib import Path

# ============================================================
# CREATE BLIND HUMAN EVALUATION FILE
# ============================================================

input_path = Path("../evaluation/reply_judge_input.csv")
output_path = Path("../evaluation/human_judge.csv")

df = pd.read_csv(input_path)

# Keep only the information needed for human evaluation
human_df = df[
    [
        "root_tweet_id",
        "customer_text",
        "golden_intent",
        "predicted_intent",
        "reply",
        "retrieved_evidence"
    ]
].copy()

# Add blank human-rating columns
human_df["human_correctness"] = ""
human_df["human_groundedness"] = ""
human_df["human_relevance"] = ""
human_df["human_tone"] = ""
human_df["human_actionability"] = ""
human_df["human_notes"] = ""

# Save
human_df.to_csv(output_path, index=False)

print("=" * 60)
print("HUMAN EVALUATION FILE CREATED")
print("=" * 60)
print(f"Examples: {len(human_df)}")
print(f"Saved to: {output_path}")

print("\nRate every reply from 1–5:")
print("1 = Very poor")
print("2 = Poor")
print("3 = Acceptable")
print("4 = Good")
print("5 = Excellent")

print("\nCriteria:")
print("Correctness   = Does the reply correctly address the customer's issue?")
print("Groundedness  = Is the reply supported by the retrieved historical evidence?")
print("Relevance     = Is the reply directly relevant to the customer's message?")
print("Tone          = Is it professional, polite, and appropriate?")
print("Actionability = Does it give the customer a useful next step?")

HUMAN EVALUATION FILE CREATED
Examples: 38
Saved to: ..\evaluation\human_judge.csv

Rate every reply from 1–5:
1 = Very poor
2 = Poor
3 = Acceptable
4 = Good
5 = Excellent

Criteria:
Correctness   = Does the reply correctly address the customer's issue?
Groundedness  = Is the reply supported by the retrieved historical evidence?
Relevance     = Is the reply directly relevant to the customer's message?
Tone          = Is it professional, polite, and appropriate?
Actionability = Does it give the customer a useful next step?


In [ ]:
from pathlib import Path

app_code = r'''
import streamlit as st
import pandas as pd
from pathlib import Path

# ============================================================
# CONFIG
# ============================================================

INPUT = Path("evaluation/reply_judge_input.csv")
OUTPUT = Path("evaluation/human_judge.csv")

criteria = {
    "human_correctness": "Correctness",
    "human_groundedness": "Groundedness",
    "human_relevance": "Relevance",
    "human_tone": "Tone",
    "human_actionability": "Actionability"
}

descriptions = {
    "human_correctness":
        "Does the reply correctly address the customer's issue?",

    "human_groundedness":
        "Is the reply supported by the retrieved historical evidence?",

    "human_relevance":
        "Is the reply directly relevant to the customer's message?",

    "human_tone":
        "Is the reply professional, polite, and appropriate?",

    "human_actionability":
        "Does the reply give the customer a useful next step?"
}

# ============================================================
# LOAD DATA
# ============================================================

if not INPUT.exists():
    st.error(f"Input file not found: {INPUT}")
    st.stop()

source = pd.read_csv(INPUT)

# Create human evaluation file if it doesn't exist
if OUTPUT.exists() and OUTPUT.stat().st_size > 0:
    df = pd.read_csv(OUTPUT)
else:
    df = source[
        [
            "root_tweet_id",
            "customer_text",
            "golden_intent",
            "predicted_intent",
            "reply",
            "retrieved_evidence"
        ]
    ].copy()

    for col in criteria:
        df[col] = pd.NA

    df["human_notes"] = ""

    df.to_csv(OUTPUT, index=False)

# ============================================================
# SESSION STATE
# ============================================================

if "index" not in st.session_state:
    st.session_state.index = 0

if "saved" not in st.session_state:
    st.session_state.saved = 0

idx = st.session_state.index

# Skip already-rated examples
while idx < len(df):
    row = df.iloc[idx]

    if all(
        pd.notna(row[col]) and str(row[col]).strip() != ""
        for col in criteria
    ):
        idx += 1
    else:
        break

st.session_state.index = idx

# ============================================================
# HEADER
# ============================================================

st.title("SupportIQ — Human Reply Evaluation")

completed = sum(
    df["human_correctness"].notna()
)

st.progress(
    min(completed / len(df), 1.0),
    text=f"Completed: {completed}/{len(df)}"
)

if idx >= len(df):
    st.success("🎉 All 38 replies have been rated!")
    st.write(f"Saved to: `{OUTPUT}`")
    st.stop()

row = df.iloc[idx]

st.subheader(f"Example {idx + 1} / {len(df)}")

st.caption(f"ID: {row['root_tweet_id']}")

# ============================================================
# CUSTOMER
# ============================================================

st.markdown("### 👤 Customer message")

st.info(str(row["customer_text"]))

# ============================================================
# INTENT CONTEXT
# ============================================================

col1, col2 = st.columns(2)

with col1:
    st.markdown("**Golden intent**")
    st.write(row["golden_intent"])

with col2:
    st.markdown("**Predicted intent**")
    st.write(row["predicted_intent"])

# ============================================================
# GENERATED REPLY
# ============================================================

st.markdown("### 🤖 Generated reply")

st.success(str(row["reply"]))

# ============================================================
# RETRIEVED EVIDENCE
# ============================================================

with st.expander("View retrieved historical evidence"):
    st.code(str(row["retrieved_evidence"]))

st.divider()

# ============================================================
# RATINGS
# ============================================================

st.markdown("## Rate this reply")

ratings = {}

for col, name in criteria.items():

    st.markdown(f"**{name}**")

    st.caption(descriptions[col])

    ratings[col] = st.radio(
        label=name,
        options=[1, 2, 3, 4, 5],
        horizontal=True,
        key=f"{col}_{idx}",
        index=None
    )

# ============================================================
# NOTES
# ============================================================

notes = st.text_area(
    "Optional notes",
    key=f"notes_{idx}",
    placeholder="Why did you give these scores?"
)

# ============================================================
# SAVE
# ============================================================

if st.button("💾 Save & Next", type="primary"):

    missing = [
        name
        for col, name in criteria.items()
        if ratings[col] is None
    ]

    if missing:
        st.warning(
            "Please rate: " + ", ".join(missing)
        )
        st.stop()

    # Save ratings
    for col in criteria:
        df.loc[idx, col] = ratings[col]

    df.loc[idx, "human_notes"] = notes

    df.to_csv(OUTPUT, index=False)

    st.session_state.index += 1
    st.rerun()
'''

Path("../evaluation").mkdir(exist_ok=True)

app_path = Path("../evaluation/human_eval_app.py")
app_path.write_text(app_code, encoding="utf-8")

print("=" * 60)
print("HUMAN EVALUATION TOOL CREATED")
print("=" * 60)
print(f"File: {app_path}")
print()
print("Run this in your VS Code terminal:")
print()
print("streamlit run ../evaluation/human_eval_app.py")

HUMAN EVALUATION TOOL CREATED
File: ..\evaluation\human_eval_app.py

Run this in your VS Code terminal:

streamlit run ../evaluation/human_eval_app.py
